# Time Series

In [2]:
import numpy as np

We will use Kalman filters to create our forecasting model. Refer to the documentation for more clarity on the algorithm.

In [12]:
class KalmanFilter:
    def __init__(self, 
                 transition_mat,
                 observation_mat,
                 noise_mat,
                 measured_covs,
                 measurements):
        self.transition_mat = transition_mat # state transition matrix
        self.observation_mat = observation_mat # observation matrix, will most likely be I
        self.noise_mat = noise_mat # process noise
        self.measured_covs = measured_covs # covariance matrices across weeks; note that week 0 is a prediction from the neural network
        self.measurements = measurements # z variables in the documentation, 7 by n array

    def check(self):
        # first check, transition matrix
        if self.transition_mat.shape[0] != self.transition_mat.shape[1]:
            return False
        else:
            dim = self.transition_mat.shape[0]

        # second check, observation matrix
        if self.observation_mat.shape[0] != self.observation_mat.shape[1] or self.observation_mat.shape[0] != dim:
            return False

        # third check, noise matrix
        if self.noise_mat.shape[0] != self.noise_mat.shape[1] or self.noise_mat.shape[0] != dim:
            return False

        # fourth check, measured covariance matrices
        measured_weeks = self.measurements.shape[0]
        if self.measured_covs.shape[0] != measured_weeks or self.measured_covs.shape[1] != self.measured_covs.shape[2] or self.measured_covs.shape[1] != dim:
            return False

    def make_predictions(self):
        measured_weeks = self.measurements.shape[0] # starts with week 0
        num_features = self.measurements.shape[1]

        # more matrix initializations
        uncertainty_mats = np.zeros((measured_weeks + 1, measured_weeks, num_features, num_features)) # first two dimensions for the two subscripts, and then each entry is a two by two square matrix
        state_mats = np.zeros((measured_weeks + 1, measured_weeks, num_features)) # first two dimensions for the two subscripts, and the entry is a 1D array
        kalman_gains = np.zeros((measured_weeks, num_features, num_features))  # 1D array of 2 by 2 matrices

        state_mats[0][0] = self.measurements[0]
        
        for i in range(measured_weeks):
            uncertainty_mats[i][i] = self.measured_covs[i]

        # per-week updates
        for week in range(measured_weeks):
            state_mats[week+1][week] = self.transition_mat @ state_mats[week][week]

            uncertainty_mats[week+1][week] = self.transition_mat @ uncertainty_mats[week][week] @ self.transition_mat.T + self.noise_mat

            if week < measured_weeks - 1:
                kalman_gains[week+1] = uncertainty_mats[week+1][week] @ self.observation_mat @ np.linalg.inv(self.observation_mat @ uncertainty_mats[week+1][week] @ self.observation_mat.T + uncertainty_mats[week+1][week+1])

                state_mats[week+1][week+1] = state_mats[week+1][week] + kalman_gains[week+1] @ (self.measurements[week+1] - self.observation_mat @ state_mats[week+1][week])

        # return prediction list, week by week
        return [state_mats[week+1][week] for week in range(measured_weeks)]
    

## Test Run

From the example: https://kalmanfilter.net

In [13]:
measurements = np.array([[10_000, 200], [11_020, 202]])
covs = np.array([[[16, 0], [0, 0.25]], [[36, 0], [0, 2.25]]])
transition = np.array([[1, 5], [0, 1]])
noise = np.array([[6.25, 2.5], [2.5, 1]])
observation = np.eye(2)

test_kalman = KalmanFilter(transition, observation, noise, covs, measurements)
test_prediction = test_kalman.make_predictions()

print(test_prediction)

[array([11000.,   200.]), array([12016.50132861,   201.42604074])]
